# 01 — Data Exploration

Load GBD 2023 data + covariates, inspect structure, and plot historical DALY trends.

**Prerequisites:** GBD CSV files downloaded to `data/gbd/`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.gbd_loader import load_gbd, load_gbd_risk_factors
from src.data.covariate_loader import download_all_covariates
from src.data.preprocessor import build_modelling_dataset, interpolate_missing

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

## 1. Load GBD DALYs

In [ ]:
gbd = load_gbd("../data/gbd/gbd_dalys_rate.csv")
print(f"Shape: {gbd.shape}")
print(f"Countries: {gbd['location'].unique()}")
print(f"Causes: {gbd['cause'].unique()}")
print(f"Years: {gbd['year'].min()} – {gbd['year'].max()}")
gbd.head()

## 2. Historical DALY trends — cardiovascular diseases

In [ ]:
cvd_causes = ["ihd", "stroke", "heart_failure"]
mask = (
    (gbd["cause"].isin(cvd_causes))
    & (gbd["sex"] == "both")
    & (gbd["age_group"] == "all_ages")
)
cvd = gbd[mask]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, cause in zip(axes, cvd_causes):
    subset = cvd[cvd["cause"] == cause]
    for loc in subset["location"].unique():
        s = subset[subset["location"] == loc]
        ax.plot(s["year"], s["value"], label=loc)
        ax.fill_between(s["year"], s["lower"], s["upper"], alpha=0.1)
    ax.set_title(cause.upper())
    ax.set_xlabel("Year")
    ax.set_ylabel("DALYs per 100k")
    ax.legend(fontsize=8)

plt.suptitle("Cardiovascular DALYs (rate per 100,000), 1990–2023", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/cvd_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. All-cause overview heatmap

In [ ]:
# Latest year DALY rates: causes × countries
latest = gbd[
    (gbd["year"] == gbd["year"].max())
    & (gbd["sex"] == "both")
    & (gbd["age_group"] == "all_ages")
]
pivot = latest.pivot_table(index="cause", columns="location", values="value")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax)
ax.set_title(f"DALY rate per 100k by cause and country ({gbd['year'].max()})")
plt.tight_layout()
plt.savefig("../outputs/figures/cause_country_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Load and inspect covariates

In [ ]:
covariates = download_all_covariates(cache_dir="../data/covariates")
print(f"Shape: {covariates.shape}")
print(f"Columns: {covariates.columns.tolist()}")
print(f"\nMissing values:\n{covariates.isnull().sum()}")
covariates.head()

## 5. Build full modelling dataset

In [ ]:
df = build_modelling_dataset(gbd=gbd, covariates=covariates)
df = interpolate_missing(df)

print(f"Modelling dataset: {df.shape}")
print(f"Unique series (group_id): {df['group_id'].nunique()}")
print(f"Split distribution:\n{df['split'].value_counts()}")
df.head()

## 6. Missingness check

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Columns with missing data:")
print(missing_pct[missing_pct > 0].to_string())